## Project Description

This is an Abstract Syntax Tree (AST) analysis project created by Khalid Mihlar.

### Project Goal

I am analyzing student submissions from Assignment 3 of the CS 2420 class. All student submissions are de-identified and anonymous. The primary goal is to examine how the `size` variable, located within the `ArrayCollection` function (which extends `Collection` in Java), is interacted with, updated, and mutated across different functions. I will be creating an AST and analyzing each node to identify these interactions.


In [9]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import javalang
import pandas as pd

# Set this to the folder that contains your .java files.
# "." means the same folder as the notebook's current working directory.
ROOT_DIR = Path("./inputs/submissions")
JAVA_GLOB = "*.java"

if not ROOT_DIR.exists():
    raise FileNotFoundError(f"Folder does not exist: {ROOT_DIR.resolve()}")

java_files = sorted(ROOT_DIR.glob(JAVA_GLOB))

print(f"Looking for Java files in: {ROOT_DIR.resolve()}")
print("Found files:")
for p in java_files:
    print(" -", p.name)

if len(java_files) == 0:
    print("\nNo .java files found.")
    print("Put your five selected Java files in the folder above,")
    print("or change ROOT_DIR to the correct folder path.")
else:
    print(f"\nConfirmed {len(java_files)} Java file(s) found.")

Looking for Java files in: \\wsl.localhost\Ubuntu\home\kmihlar\Projects\ast-project\inputs\submissions
Found files:
 - AfricanWildDog.java
 - Anthozoa.java
 - BlackRhinoceros.java
 - Newt.java
 - Yak.java

Confirmed 5 Java file(s) found.


# AST extraction rubric

A use of `size` is classified as follows.

### Read
Count as a read when `size` appears in:

- return expressions, comparisons, conditions, loop bounds
- array indices like `data[size]`
- right-hand side expressions
- method arguments
- unary or compound updates, because they consult the old value

### Write
Count as a write when `size` appears as the target of:

- assignment: `size = ...`
- unary update: `size++`, `++size`, `size--`, `--size`
- compound assignment: `size += 1`, `size -= 1`

### Read + write
Count as both when syntax implies both:

- `size++`
- `size--`
- `size += 1`
- `size = size + 1`
- `size = size - 1`

### Method-level aggregation
For each constructor and method, the notebook outputs:

- `reads_size`
- `writes_size`
- `write_kinds`
- `evidence`
- `local_shadowing_of_size`




In [10]:
def load_source(path: Path) -> str:
    return path.read_text(encoding="utf-8")

def parse_java_file(path: Path):
    src = load_source(path)
    return javalang.parse.parse(src)

raw_trees = {}
parse_errors = {}

for path in java_files:
    try:
        raw_trees[path.name] = parse_java_file(path)
        print(f"Parsed successfully: {path.name}")
    except Exception as e:
        parse_errors[path.name] = str(e)
        print(f"Failed to parse: {path.name}")
        print(f"Error: {e}")
        print("-" * 50)



Parsed successfully: AfricanWildDog.java
Parsed successfully: Anthozoa.java
Parsed successfully: BlackRhinoceros.java
Parsed successfully: Newt.java
Parsed successfully: Yak.java


In [17]:
import javalang
import json

def dump_nodes(tree):
    for path, node in tree:
        print("=" * 60)
        print("NODE TYPE:", type(node).__name__)
        print("ATTRS:", getattr(node, "attrs", []))

        for attr in getattr(node, "attrs", []):
            print(f"  {attr}: {getattr(node, attr)}")

# Simply a way for me to see all the nodes and it's relevant information to get a better understanding 
def visualize_ast(node, indent=0):
    prefix = "  " * indent

    if isinstance(node, javalang.ast.Node):
        print(f"{prefix}{type(node).__name__}")

        for attr in node.attrs:
            value = getattr(node, attr)
            if value is None or value == []:
                continue

            print(f"{prefix}  .{attr}:")
            visualize_ast(value, indent + 2)

    elif isinstance(node, list):
        print(f"{prefix}list[{len(node)}]")
        for i, item in enumerate(node):
            print(f"{prefix}  [{i}]")
            visualize_ast(item, indent + 2)

    else:
        print(f"{prefix}{repr(node)}")

java_code = """
public class Example {
    public int test() {
        int size = 0;
        size++;
        return size;
    }
}
"""
print(raw_trees)

{'AfricanWildDog.java': CompilationUnit(imports=[Import(path=java.util.ArrayList, static=False, wildcard=False), Import(path=java.util.Collection, static=False, wildcard=False), Import(path=java.util.Comparator, static=False, wildcard=False), Import(path=java.util.Iterator, static=False, wildcard=False), Import(path=java.util.NoSuchElementException, static=False, wildcard=False)], package=PackageDeclaration(annotations=None, documentation=None, modifiers=None, name=assignment3), types=[ClassDeclaration(annotations=[], body=[FieldDeclaration(annotations=[], declarators=[VariableDeclarator(dimensions=[None], initializer=None, name=data)], documentation=None, modifiers=set(), type=ReferenceType(arguments=None, dimensions=[], name=T, sub_type=None)), FieldDeclaration(annotations=[], declarators=[VariableDeclarator(dimensions=[], initializer=None, name=size)], documentation=None, modifiers=set(), type=BasicType(dimensions=[], name=int)), FieldDeclaration(annotations=[], declarators=[Variabl

In [22]:

# Simply will return nodes with information that contain either the name or member size, along with the parent path, node type, and operators based on testing
def find_size_nodes(tree):
    matches = []

    for path, node in tree:
        is_size_node = False
        if hasattr(node, "name") and getattr(node, "name") == "size":
            is_size_node = True
        elif hasattr(node, "member") and getattr(node, "member") == "size":
            is_size_node = True
        if is_size_node:
            matches.append((path, node))

    print("Number of matching nodes:", len(matches))

    for i, (path, node) in enumerate(matches, 1):
        print("\n" + "=" * 60)
        print(f"Match {i}")
        print("Node type:", type(node).__name__)
        print("Attributes:")

        for attr in getattr(node, "attrs", []):
            print(f"  {attr}: {getattr(node, attr)}")

        print("Parent path:")
        for parent in path:
            if hasattr(parent, "__class__"):
                print("  ", type(parent).__name__)

def find_size_nodes_in_all_trees(raw_trees):
    results = {}

    for filename, tree in raw_trees.items():
        results[filename] = find_size_nodes(tree)

    return results

stuff = find_size_nodes_in_all_trees(raw_trees)

Number of matching nodes: 15

Match 1
Node type: VariableDeclarator
Attributes:
  name: size
  dimensions: []
  initializer: None
Parent path:
   CompilationUnit
   list
   ClassDeclaration
   list
   FieldDeclaration
   list

Match 2
Node type: MemberReference
Attributes:
  prefix_operators: []
  postfix_operators: []
  qualifier: 
  selectors: []
  member: size
Parent path:
   CompilationUnit
   list
   ClassDeclaration
   list
   ConstructorDeclaration
   list
   StatementExpression
   Assignment

Match 3
Node type: MemberReference
Attributes:
  prefix_operators: []
  postfix_operators: []
  qualifier: 
  selectors: []
  member: size
Parent path:
   CompilationUnit
   list
   ClassDeclaration
   list
   MethodDeclaration
   list
   IfStatement
   BinaryOperation

Match 4
Node type: MemberReference
Attributes:
  prefix_operators: []
  postfix_operators: []
  qualifier: 
  selectors: []
  member: size
Parent path:
   CompilationUnit
   list
   ClassDeclaration
   list
   MethodDeclara